# Notebook 05 — Hybrid M3: Structured {name, strength, unit} Decoding + Clinical Metrics

In [1]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
import re, random
from dataclasses import dataclass
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd
from PIL import Image
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader

DEVICE = (torch.device("mps") if torch.backends.mps.is_available()
          else torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))
print(f"Using device: {DEVICE} | torch {torch.__version__}")

Using device: mps | torch 2.12.0


In [2]:
@dataclass
class Config:
    data_root: Path = Path("../data/pharmacy_lk")
    train_csv: str = "splits/train.csv"
    test_csv: str = "splits/test.csv"
    img_dir: str = "images"
    img_col: str = "image_filename"
    label_col: str = "medicine_name"
    img_height: int = 48
    img_width: int = 320
    rnn_hidden: int = 256
    rnn_layers: int = 2
    dropout: float = 0.2
    batch_size: int = 64
    num_workers: int = 0
    seeds: tuple = (42, 43, 44, 45, 46)
    name_tau: float = 0.6
    formulary_path: "Path | None" = None
    multiseed_dir: Path = Path("../checkpoints/multiseed")
    out_dir: Path = Path("../checkpoints/hybrid_m3")

CFG = Config()
CFG.out_dir.mkdir(parents=True, exist_ok=True)

## 1. Shared components + the structured parser (same rules as the prep notebook)

In [3]:
class Vocab:
    BLANK = 0
    def __init__(self, texts):
        chars = sorted(set("".join(texts)))
        self.idx2char = {i + 1: c for i, c in enumerate(chars)}
        self.char2idx = {c: i for i, c in self.idx2char.items()}
    def __len__(self):
        return len(self.idx2char) + 1
    def encode(self, t):
        return [self.char2idx[c] for c in t]
    def decode_greedy(self, indices):
        out, prev = [], None
        for i in indices:
            if i != prev and i != self.BLANK:
                out.append(self.idx2char[i])
            prev = i
        return "".join(out)

class WordImageDataset(Dataset):
    def __init__(self, csv_path, img_dir, cfg, vocab=None):
        self.df = pd.read_csv(csv_path).dropna(subset=[cfg.label_col])
        self.df[cfg.label_col] = self.df[cfg.label_col].astype(str).str.strip()
        self.img_dir = Path(img_dir); self.cfg = cfg; self.vocab = vocab
    def labels(self):
        return self.df[self.cfg.label_col].tolist()
    def __len__(self):
        return len(self.df)
    def _load(self, path):
        img = Image.open(path).convert("L"); w, h = img.size
        new_w = min(max(1, int(round(w * self.cfg.img_height / h))), self.cfg.img_width)
        img = img.resize((new_w, self.cfg.img_height), Image.BILINEAR)
        canvas = Image.new("L", (self.cfg.img_width, self.cfg.img_height), color=255)
        canvas.paste(img, (0, 0)); return canvas
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        x = torch.from_numpy(np.array(self._load(self.img_dir / str(row[self.cfg.img_col])),
                                      dtype=np.float32) / 255.0).unsqueeze(0)
        return x, row[self.cfg.label_col], str(row[self.cfg.img_col])

def collate(batch):
    xs, texts, fnames = zip(*batch)
    return torch.stack(xs), list(texts), list(fnames)

def edit_distance(a, b):
    if a == b: return 0
    if not a: return len(b)
    if not b: return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        curr = [i]
        for j, cb in enumerate(b, 1):
            curr.append(min(prev[j] + 1, curr[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = curr
    return prev[-1]

DOSE_RE = re.compile(r"(?<=\s)(\d+\.?\d*)\s*(mg|mcg|ml|g|gm|iu|%|cal)\b")
def parse_structure(s):
    s = s.strip()
    m = DOSE_RE.search(s)
    if not m:
        return s, "", ""
    strength, unit = m.group(1), m.group(2)
    name = re.sub(r"\s+", " ", (s[:m.start()] + s[m.end():]).strip(" .-"))
    return name, strength, unit

class CRNN(nn.Module):
    def __init__(self, n_classes, rnn_hidden=256, rnn_layers=2, dropout=0.2):
        super().__init__()
        def conv(i, o, bn=False):
            L = [nn.Conv2d(i, o, 3, 1, 1)]
            if bn: L.append(nn.BatchNorm2d(o))
            L.append(nn.ReLU(inplace=True)); return L
        self.cnn = nn.Sequential(
            *conv(1, 64), nn.MaxPool2d(2, 2),
            *conv(64, 128), nn.MaxPool2d(2, 2),
            *conv(128, 256), *conv(256, 256), nn.MaxPool2d((2, 1), (2, 1)),
            *conv(256, 512, bn=True), *conv(512, 512, bn=True), nn.MaxPool2d((2, 1), (2, 1)),
        )
        self.collapse = nn.AdaptiveAvgPool2d((1, None))
        self.rnn = nn.LSTM(512, rnn_hidden, rnn_layers, bidirectional=True,
                           dropout=dropout if rnn_layers > 1 else 0.0)
        self.head = nn.Linear(2 * rnn_hidden, n_classes)
    def forward(self, x):
        f = self.collapse(self.cnn(x)).squeeze(2).permute(2, 0, 1)
        seq, _ = self.rnn(f)
        return self.head(seq)

## 2. Name-field lexicon (training NAMES only, dosage stripped)

In [4]:
train_df = pd.read_csv(CFG.data_root / CFG.train_csv)
train_names = set(train_df["name"].dropna().astype(str).str.strip().str.lower())
train_names.discard("")
if CFG.formulary_path and Path(CFG.formulary_path).exists():
    train_names |= {l.strip().lower() for l in open(CFG.formulary_path) if l.strip()}
name_lexicon = sorted(train_names)
lex_by_len = defaultdict(list)
for w in name_lexicon:
    lex_by_len[len(w)].append(w)
print(f"name lexicon size: {len(name_lexicon)}")

KNOWN_UNITS = {"mg", "mcg", "ml", "g", "gm", "iu", "%", "cal"}

def norm_strength(s):
    """Canonicalise a strength value so '625', '625.0', 625.0 all compare equal.
    Returns '' for empty/NaN. Integers lose the trailing '.0'."""
    if s is None:
        return ""
    s = str(s).strip()
    if s == "" or s.lower() == "nan":
        return ""
    try:
        f = float(s)
        return str(int(f)) if f == int(f) else str(f)   # 625.0 -> '625', 2.5 -> '2.5'
    except ValueError:
        return s

def nearest_name(word, max_len_gap=3):
    if not word:
        return None, 10**9
    if word in lex_by_len.get(len(word), ()):
        return word, 0
    best, best_d = None, 10**9
    for L in range(len(word) - max_len_gap, len(word) + max_len_gap + 1):
        for entry in lex_by_len.get(L, ()):
            d = edit_distance(word, entry)
            if d < best_d:
                best, best_d = entry, d
                if d == 1:
                    return best, best_d
    return best, best_d

def structured_decode(raw_pred, tau):
    name, strength, unit = parse_structure(raw_pred)
    entry, d = nearest_name(name)
    if entry is not None and d / max(len(name), 1) <= tau:
        name = entry
    unit = unit if unit in KNOWN_UNITS else ""
    return {"name": name, "strength": norm_strength(strength), "unit": unit}

name lexicon size: 1267


## 3. Field-level + clinical metrics
right_drug_wrong_dose = name correct BUT strength wrong (the dangerous class; want LOW).

In [5]:
def field_metrics(pred_fields, gt_fields):
    n = len(gt_fields)
    name_acc = sum(p["name"] == g["name"] for p, g in zip(pred_fields, gt_fields)) / n
    dose_idx = [i for i, g in enumerate(gt_fields) if g["strength"] != ""]
    out = {"name_acc": name_acc, "n_all": n, "n_dose": len(dose_idx)}
    if dose_idx:
        out["strength_acc"] = sum(pred_fields[i]["strength"] == gt_fields[i]["strength"] for i in dose_idx) / len(dose_idx)
        out["unit_acc"] = sum(pred_fields[i]["unit"] == gt_fields[i]["unit"] for i in dose_idx) / len(dose_idx)
        out["full_acc"] = sum(all(pred_fields[i][k] == gt_fields[i][k] for k in ("name", "strength", "unit"))
                              for i in dose_idx) / len(dose_idx)
        out["right_drug_wrong_dose"] = sum((pred_fields[i]["name"] == gt_fields[i]["name"]) and
                                           (pred_fields[i]["strength"] != gt_fields[i]["strength"])
                                           for i in dose_idx) / len(dose_idx)
    return out

## 4. Evaluate across trained seeds (reuses Notebook 04 checkpoints if present)

In [6]:
def greedy_raw(model, loader, vocab):
    model.eval(); preds, refs, files = [], [], []
    with torch.no_grad():
        for xb, texts, fnames in loader:
            logits = model(xb.to(DEVICE))
            idx = logits.argmax(-1).permute(1, 0).cpu()
            preds += [vocab.decode_greedy(s.tolist()) for s in idx]
            refs += texts; files += fnames
    return preds, refs, files

_train_labels = pd.read_csv(CFG.data_root / CFG.train_csv)[CFG.label_col].dropna().astype(str).str.strip().tolist()
VOCAB = Vocab(_train_labels)
test_ds = WordImageDataset(CFG.data_root / CFG.test_csv, CFG.data_root / CFG.img_dir, CFG, vocab=VOCAB)
test_dl = DataLoader(test_ds, CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, collate_fn=collate)

test_meta = pd.read_csv(CFG.data_root / CFG.test_csv)
gt_by_file = {}
for _, r in test_meta.iterrows():
    gt_by_file[str(r[CFG.img_col])] = {"name": str(r["name"]).strip().lower(),
                                       "strength": norm_strength(r["strength"]),
                                       "unit": "" if pd.isna(r["unit"]) else str(r["unit"]).strip()}

seed_ckpts = {s: CFG.multiseed_dir / f"seed{s}_best.pt" for s in CFG.seeds}
available = {s: p for s, p in seed_ckpts.items() if p.exists()}
single_ckpt = Path("../checkpoints/baseline_crnn_v2/best.pt")

if available:
    eval_targets = list(available.items())
    print(f"using per-seed checkpoints: {sorted(available)}")
else:
    eval_targets = [("single", single_ckpt)]
    print("per-seed weights not found — evaluating one representative checkpoint:")
    print(f"  {single_ckpt}")
    print("  (M3 is an evaluation layer; a single-seed result is a valid demonstration.)")

rows = []
for tag, ckpt_path in eval_targets:
    if not Path(ckpt_path).exists():
        print(f"  [skip] {tag}: checkpoint missing at {ckpt_path}")
        continue
    ck = torch.load(ckpt_path, map_location="cpu")
    state = ck["model"] if "model" in ck else ck
    model = CRNN(len(VOCAB), CFG.rnn_hidden, CFG.rnn_layers, CFG.dropout)
    model.load_state_dict(state); model.to(DEVICE)
    raw, refs, files = greedy_raw(model, test_dl, VOCAB)
    pred_fields = [structured_decode(p, CFG.name_tau) for p in raw]
    gt_fields = [gt_by_file[f] for f in files]
    fm = field_metrics(pred_fields, gt_fields); fm["tag"] = tag
    rows.append(fm)
    msg = f"  {tag}: name_acc={fm['name_acc']:.4f}"
    if fm["n_dose"]:
        msg += (f" | strength_acc={fm.get('strength_acc', float('nan')):.3f}"
                f" | right_drug_wrong_dose={fm.get('right_drug_wrong_dose', float('nan')):.3f}"
                f" (n_dose={fm['n_dose']})")
    print(msg)

fields_df = pd.DataFrame(rows)
fields_df.to_csv(CFG.out_dir / "m3_field_metrics.csv", index=False)

per-seed weights not found — evaluating one representative checkpoint:
  ../checkpoints/baseline_crnn_v2/best.pt
  (M3 is an evaluation layer; a single-seed result is a valid demonstration.)
  single: name_acc=0.3056 | strength_acc=0.000 | right_drug_wrong_dose=0.000 (n_dose=29)


## 5. Summary

In [7]:
def msd(col):
    if col not in fields_df or fields_df[col].dropna().empty:
        return "n/a"
    v = fields_df[col].dropna()
    return f"{v.mean():.4f} ± {v.std(ddof=1):.4f}" if len(v) > 1 else f"{v.iloc[0]:.4f}"

print(f"evaluated on {len(fields_df)} checkpoint(s); dosage subset n_dose="
      f"{int(fields_df['n_dose'].iloc[0]) if len(fields_df) else 0}")
print("\nFIELD-LEVEL & CLINICAL METRICS:")
for label, col in [("Name accuracy (all test)", "name_acc"),
                   ("Strength accuracy (dosage subset)", "strength_acc"),
                   ("Unit accuracy (dosage subset)", "unit_acc"),
                   ("Full name+strength+unit accuracy", "full_acc"),
                   ("CLINICAL right-drug-wrong-dose (lower better)", "right_drug_wrong_dose")]:
    print(f"  {label:46s}: {msd(col)}")

evaluated on 1 checkpoint(s); dosage subset n_dose=29

FIELD-LEVEL & CLINICAL METRICS:
  Name accuracy (all test)                      : 0.3056
  Strength accuracy (dosage subset)             : 0.0000
  Unit accuracy (dosage subset)                 : 0.0000
  Full name+strength+unit accuracy              : 0.0000
  CLINICAL right-drug-wrong-dose (lower better) : 0.0000


## 6. Qualitative: parsed fields for dosage-bearing test cases (thesis table)

In [8]:
if eval_targets:
    ck = torch.load(eval_targets[0][1], map_location="cpu")
    state = ck["model"] if "model" in ck else ck
    model = CRNN(len(VOCAB), CFG.rnn_hidden, CFG.rnn_layers, CFG.dropout)
    model.load_state_dict(state); model.to(DEVICE)
    raw, refs, files = greedy_raw(model, test_dl, VOCAB)
    examples = []
    for p, r, f in zip(raw, refs, files):
        g = gt_by_file[f]
        if g["strength"] != "":
            pf = structured_decode(p, CFG.name_tau)
            examples.append({"gt": r, "raw_pred": p, "pred_name": pf["name"], "gt_name": g["name"],
                             "pred_dose": f"{pf['strength']}{pf['unit']}", "gt_dose": f"{g['strength']}{g['unit']}",
                             "name_ok": pf["name"] == g["name"], "dose_ok": pf["strength"] == g["strength"]})
    ex_df = pd.DataFrame(examples)
    ex_df.to_csv(CFG.out_dir / "m3_dosage_examples.csv", index=False)
    print(f"dosage-bearing test cases: {len(ex_df)}")
    if len(ex_df):
        print(ex_df.head(20).to_string(index=False))

dosage-bearing test cases: 29
                 gt   raw_pred pred_name        gt_name pred_dose gt_dose  name_ok  dose_ok
   zerodol cr 200mg    esinnmg     esome     zerodol cr             200mg    False    False
       brufen 400mg   onistamg    festam         brufen             400mg    False    False
    augmentin 625mg    esrsamg     curam      augmentin             625mg    False    False
spironolactone 25mg    sososmg     esome spironolactone              25mg    False    False
         happi 20mg      alomg      alos          happi              20mg    False    False
       losacar 50mg       esmg        es        losacar              50mg    False    False
      pantocid 40mg    risismg   frisium       pantocid              40mg    False    False
        hexilon 4mg     rulomg    amlong        hexilon               4mg    False    False
    augmentin 625mg      esomg     esome      augmentin             625mg    False    False
      allegra 180mg     sosrmg    sosrmg        al

## 7. For the thesis
- Headline stays M2 (multi-seed, significant). M3 is the SECONDARY clinical layer.
- Report field metrics WITH n (name over all 792; strength/unit/clinical over ~29 dosage rows).
- right_drug_wrong_dose is the novel clinically-meaningful metric; argue why CER/WER cannot
  express it and why a plausible-but-wrong dose is a hard-to-catch hazard.
- Limitation/future work: larger dosage-annotated corpus + external formulary.